# Lecture 9.3 — Building the AuditPlugin

**Section 09 — System-Wide Control with Plugins**

This notebook is a **live coding** notebook. Cells marked `[PRE-BUILT]` are complete and ready to run. Cells marked `[LIVE]` are where the `AuditPlugin` class, its registration on the `App`, and the audit trail inspection get built during recording.

This is the `Lecture_9_3_AuditPlugin-SOLUTION.ipynb` file. Every cell, including the live ones, is filled in completely. This is the version to download and keep.

## Cell 1 — Install the Google ADK Package [PRE-BUILT]

This notebook uses the Google Agent Development Kit (ADK), Google's Python framework for building and running LLM-backed agents. The cell below installs it.

The version is pinned to `google-adk==2.6.3` so that every example in this notebook behaves the same way it does in the recording, regardless of what has shipped on PyPI since. If the package is already present in this Colab session, the install completes almost instantly and moves on.

| Comment in the cell | Meaning |
|---|---|
| Reproducibility pin | Keeps this notebook's behaviour stable over time |
| `pip install google-adk` (no version) | Removes the pin, installs whatever is newest |
| Substitute your own version | Swap `2.6.3` for any version you prefer to test against |

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install google-adk
# Or substitute your preferred version below.
!pip install google-adk==2.6.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 11.9 MB/s eta 0:00:00


## Cell 2 — Configure the Google API Key [PRE-BUILT]

Every agent in this notebook calls the Gemini API, which needs an API key. This course uses **Google Colab Secrets** exclusively, so the key never appears in plain text in the notebook itself.

**To add the secret in Colab:**
1. Click the key icon (🔑) in the left sidebar of this Colab notebook.
2. Click **Add new secret**.
3. Set the name to `GOOGLE_API_KEY` and paste in a key generated from [Google AI Studio](https://aistudio.google.com/apikey).
4. Toggle **Notebook access** on for this notebook.

**Running locally instead of Colab?** Set `GOOGLE_API_KEY` as an environment variable in your terminal before starting Jupyter, rather than using `userdata.get`.

In [2]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

## Cell 3 — Declare the Model Name [PRE-BUILT]

Every agent defined later in this notebook references `MODEL_NAME` instead of a hardcoded model string. Changing the value in this one cell updates the model used by every agent in the notebook at once.

| Variable | Value | Purpose |
|---|---|---|
| `MODEL_NAME` | `"gemini-3.7-flash"` | The model used by every agent defined below |

In [3]:
# See latest models at: https://ai.google.dev/gemini-api/docs/models
MODEL_NAME = "gemini-3.7-flash"

## Cell 4 — Define the Custom Tools [PRE-BUILT]

The Autonomous Event Genie relies on six plain Python functions instead of one monolithic tool. Each one gives a single agent a narrow, well-defined capability: writing to session state, summing costs, or breaking out of a loop. `GUEST_DATABASE` is a simple in-memory list standing in for a real datastore, and `COMPLETION_PHRASE` is the exact string `accountant_agent` writes when a plan is on budget, which `cost_cutter_agent` checks for before calling `exit_loop`.

| Tool | Used by | Purpose |
|---|---|---|
| `update_session_state` | `intake_agent` | Writes `event_type`, `city`, and `budget` to session state |
| `add_guest`, `get_guest_list` | `guest_management_agent` | Manage the in-memory guest list |
| `sum_costs` | `accountant_agent` | Totals a list of costs |
| `exit_loop` | `cost_cutter_agent` | Sets `tool_context.actions.escalate = True` to stop the budget loop |
| `send_mock_email` | `master_orchestrator_agent` | Simulates sending the drafted announcement email |

Four of these six functions take a `tool_context` parameter. That is how a plain Python function reaches into session state, or in the case of `exit_loop`, sets the `escalate` action that tells a `LoopAgent` to stop.

In [4]:
from google.adk.tools import ToolContext

GUEST_DATABASE = []

COMPLETION_PHRASE = "The plan is within the budget."


def add_guest(name: str, email: str) -> dict:
    """Adds a guest to the guest database."""
    GUEST_DATABASE.append({"name": name, "email": email})
    return {"status": "success", "guest": name}


def get_guest_list(tool_context: ToolContext) -> dict:
    """Retrieves the current guest list and writes it to session state."""
    tool_context.state["guest_list"] = GUEST_DATABASE
    return {"guest_list": GUEST_DATABASE}


def sum_costs(costs: list[float]) -> float:
    """Sums a list of costs."""
    return sum(costs)


def exit_loop(tool_context: ToolContext) -> dict:
    """Signals the budget refinement loop to stop iterating."""
    tool_context.actions.escalate = True
    return {"status": "loop_exited"}


def update_session_state(
    tool_context: ToolContext,
    event_type: str,
    city: str,
    budget: float,
) -> dict:
    """Writes the extracted event details to session state."""
    tool_context.state["event_type"] = event_type
    tool_context.state["city"] = city
    tool_context.state["budget"] = budget
    return {"event_type": event_type, "city": city, "budget": budget}


def send_mock_email(tool_context: ToolContext) -> dict:
    """Simulates sending the drafted announcement email to every guest."""
    email_draft = tool_context.state.get("email_draft", {})
    recipients = [guest["email"] for guest in GUEST_DATABASE]
    return {
        "status": "sent",
        "subject": email_draft.get("subject", ""),
        "recipient_count": len(recipients),
    }

## Cell 5 — Wrap Google Search as a Sub-Agent Tool [PRE-BUILT]

`google_search` is a built-in ADK tool, but ADK only allows it on an agent that has no other tools attached. `venue_scout_agent`, `catering_scout_agent`, and `entertainment_scout_agent` attach `google_search` directly, since search is their only tool. `cost_cutter_agent` needs both search and `exit_loop` at the same time, so instead of attaching `google_search` directly, it calls a small dedicated search agent through an `AgentTool` wrapper. The wrapped agent does nothing but search, which sidesteps the one-tool restriction without changing what the search itself returns.

In [5]:
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool

google_search_agent = Agent(
    name="Google_Search_Agent",
    model=MODEL_NAME,
    instruction="You are just a wrapper for the Google Search tool.",
    tools=[google_search]
)
google_search_tool = AgentTool(agent=google_search_agent)

## Cell 6 — Define the Ten Specialist Agents [PRE-BUILT]

This is the full specialist layer of the Autonomous Event Genie, unchanged from Section 6. `communications_agent` needs a strict output shape, so an `EmailDraft` pydantic model is defined first and passed in as that agent's `output_schema`.

| Agent | Role | Writes to state via `output_key` |
|---|---|---|
| `intake_agent` | Extracts event type, city, budget | (writes via `update_session_state` tool instead) |
| `guest_management_agent` | Manages the guest list | (writes via `get_guest_list` tool instead) |
| `venue_scout_agent` | Finds 3 venues with costs | `venue_options` |
| `catering_scout_agent` | Finds 3 caterers with costs | `catering_options` |
| `entertainment_scout_agent` | Finds 3 entertainment options with costs | `entertainment_options` |
| `initial_plan_synthesizer_agent` | Combines the three scout results into one plan | `current_plan` |
| `accountant_agent` | Picks the cheapest options, checks against budget | `evaluation` |
| `cost_cutter_agent` | Finds cheaper alternatives if over budget | `current_plan` |
| `communications_agent` | Drafts the announcement email | `email_draft` |
| `final_report_agent` | Compiles the final markdown report | (returns final text directly) |

None of these ten agents have any callback attached. That clean slate is exactly what makes the payoff of this lecture visible: adding one `AuditPlugin` on the `App` covers all ten of them (and the six agents still to come) without touching a single one of these definitions.

In [6]:
from pydantic import BaseModel, Field


class EmailDraft(BaseModel):
    subject: str = Field(description="The compelling subject line for the email.")
    body: str = Field(description="The full, well-formatted body of the email.")


intake_agent = Agent(
    name="intake_agent",
    model=MODEL_NAME,
    instruction="From the user's query, identify the event type, city, and budget. Then call update_session_state.",
    tools=[update_session_state]
)

guest_management_agent = Agent(
    name="guest_management_agent",
    model=MODEL_NAME,
    instruction="You are a guest management assistant. Use add_guest and get_guest_list tools.",
    tools=[add_guest, get_guest_list]
)

venue_scout_agent = Agent(
    name="venue_scout_agent",
    model=MODEL_NAME,
    instruction='Find 3 venues for {event_type} in {city} with costs. Output JSON: {"venues": [...]}',
    tools=[google_search],
    output_key="venue_options"
)

catering_scout_agent = Agent(
    name="catering_scout_agent",
    model=MODEL_NAME,
    instruction='Find 3 caterers for {event_type} in {city} with costs. Output JSON: {"caterers": [...]}',
    tools=[google_search],
    output_key="catering_options"
)

entertainment_scout_agent = Agent(
    name="entertainment_scout_agent",
    model=MODEL_NAME,
    instruction='Find 3 entertainment options for {event_type} in {city} with costs. Output JSON: {"entertainment": [...]}',
    tools=[google_search],
    output_key="entertainment_options"
)

initial_plan_synthesizer_agent = Agent(
    name="initial_plan_synthesizer_agent",
    model=MODEL_NAME,
    instruction="Combine {venue_options}, {catering_options}, {entertainment_options} into one JSON with keys venues, caterers, entertainment.",
    output_key="current_plan"
)

accountant_agent = Agent(
    name="accountant_agent",
    model=MODEL_NAME,
    tools=[sum_costs],
    instruction="Select cheapest option from each category in {current_plan}. Use sum_costs. If total > {budget}, output JSON with critique and cheapest_plan. Else output JSON with completion phrase and cheapest_plan.",
    output_key="evaluation"
)

cost_cutter_agent = Agent(
    name="cost_cutter_agent",
    model=MODEL_NAME,
    tools=[google_search_tool, exit_loop],
    instruction="Check {evaluation} critique. If completion phrase, call exit_loop. Else find cheaper alternative for the flagged item. Output updated plan JSON.",
    output_key="current_plan"
)

communications_agent = Agent(
    name="communications_agent",
    model=MODEL_NAME,
    instruction='Draft announcement email from {current_plan}. Output raw JSON only: {"subject": ..., "body": ...}',
    output_key="email_draft",
    output_schema=EmailDraft
)

final_report_agent = Agent(
    name="final_report_agent",
    model=MODEL_NAME,
    instruction="Compile final event plan markdown report using {current_plan}, {venue_options}, {catering_options}, {entertainment_options}, {budget}. Include executive summary, approved plan, alternatives, next steps."
)

## Cell 7 — Define the Five Workflow Agents [PRE-BUILT]

These five workflow agents wire the ten specialists together into a single pipeline. `ParallelAgent`, `SequentialAgent`, and `LoopAgent` are ADK's built-in workflow primitives.

| Workflow agent | Type | Role |
|---|---|---|
| `parallel_logistics_scout` | `ParallelAgent` | Runs the three scout agents simultaneously |
| `initial_planning_workflow` | `SequentialAgent` | Scouts, then the synthesizer |
| `budget_refinement_loop` | `LoopAgent` | Alternates `accountant_agent` and `cost_cutter_agent` until on-budget or 3 iterations |
| `budget_optimizer_workflow` | `SequentialAgent` | Wraps the loop |
| `full_plan_workflow` | `SequentialAgent` | Intake, then planning, then budget, then the final report |

**A note on deprecation warnings:** ADK 2.6 introduced a new `Workflow` graph API as the eventual replacement for `SequentialAgent`, `ParallelAgent`, and `LoopAgent`. The ADK documentation currently states that `Workflow` cannot yet be used as an `LlmAgent` sub-agent, which is exactly how `master_orchestrator_agent` consumes these workflow agents via `AgentTool` in Cell 8. Migrating now would mean restructuring the orchestrator beyond the scope of this section, so any deprecation warning these classes raise is safe to ignore for now.

In [7]:
from google.adk.agents import ParallelAgent, SequentialAgent, LoopAgent

parallel_logistics_scout = ParallelAgent(
    name="parallel_logistics_scout",
    sub_agents=[venue_scout_agent, catering_scout_agent, entertainment_scout_agent]
)

initial_planning_workflow = SequentialAgent(
    name="initial_planning_workflow",
    sub_agents=[parallel_logistics_scout, initial_plan_synthesizer_agent]
)

budget_refinement_loop = LoopAgent(
    name="budget_refinement_loop",
    sub_agents=[accountant_agent, cost_cutter_agent],
    max_iterations=3
)

budget_optimizer_workflow = SequentialAgent(
    name="budget_optimizer_workflow",
    sub_agents=[budget_refinement_loop]
)

full_plan_workflow = SequentialAgent(
    name="full_plan_workflow",
    sub_agents=[
        intake_agent,
        initial_planning_workflow,
        budget_optimizer_workflow,
        final_report_agent
    ]
)

/tmp/ipykernel_416/2137418663.py:3: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  parallel_logistics_scout = ParallelAgent(
/tmp/ipykernel_416/2137418663.py:8: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  initial_planning_workflow = SequentialAgent(
/tmp/ipykernel_416/2137418663.py:13: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  budget_refinement_loop = LoopAgent(
/tmp/ipykernel_416/2137418663.py:19: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  budget_optimizer_workflow = SequentialAgent(
/tmp/ipykernel_416/213

## Cell 8 — Define the Master Orchestrator [PRE-BUILT]

`master_orchestrator_agent` is the single agent a user actually talks to. It never plans, sources venues, or drafts email itself. Its only job is to read the user's request and delegate to the right tool: `full_plan_workflow_tool` for planning, `guest_list_manager_tool` for guest management, `communications_tool` for drafting, and the plain `send_mock_email` function for sending.

Everything underneath it — all ten specialist agents and five workflow agents — is invisible to the user. They only ever see this one agent's replies. Together with `master_orchestrator_agent` itself, that is the full 16-agent system this lecture's `AuditPlugin` will cover.

In [8]:
full_plan_workflow_tool = AgentTool(agent=full_plan_workflow)
guest_list_manager_tool = AgentTool(agent=guest_management_agent)
communications_tool = AgentTool(agent=communications_agent)

master_orchestrator_agent = Agent(
    name="master_orchestrator_agent",
    model=MODEL_NAME,
    instruction="Delegate: full_plan_workflow_tool for planning, guest_list_manager_tool for guests, communications_tool for email drafting, send_mock_email for sending.",
    tools=[
        full_plan_workflow_tool,
        guest_list_manager_tool,
        communications_tool,
        send_mock_email
    ]
)

## Cell 9 — Session Service and the Run Helper [PRE-BUILT]

`InMemorySessionService` holds this conversation in process memory for the lifetime of this notebook run. Nothing here persists once the runtime restarts, which is fine for this demo but not for production use. Section 11 later in this course swaps in a persistent session service without changing any agent code.

`run_agent_query` wraps the standard ADK run loop. Two things worth noting about this version:

1. **`Runner(app=app, ...)`** — In ADK 2.x, Runner takes an `App` object directly rather than a bare `agent=` and `app_name=` string separately. The App carries both the root agent and any application-wide configuration including plugins.
2. **`app` is closed over, not passed in** — `app` is defined in this notebook's scope and is the same object for every call, so the helper takes only three parameters: `query`, `session`, and `user_id`. There is no need to pass `app` on every call.

| Parameter | What it is |
|---|---|
| `query` | The plain text of the user message for this turn |
| `session` | The `Session` object created before the first turn |
| `user_id` | A string identifying the user across sessions |

In [9]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

session_service = InMemorySessionService()


async def run_agent_query(query, session, user_id):
    """Creates a runner from the App and executes one query turn."""
    runner = Runner(
        app=app,
        session_service=session_service,
    )
    final_response = None
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(parts=[Part(text=query)], role="user"),
    ):
        if event.is_final_response():
            final_response = event.content.parts[0].text
    return final_response

## Cell 10 — Build the AuditPlugin Class [LIVE]

This is the entire point of this lecture. `BasePlugin` is the base class every Plugin subclasses. Overriding one of its hook methods is all it takes to observe something happening anywhere in the agent tree, across every agent in the `App`, without touching a single agent's own definition.

`self.log` is a plain Python list living on the plugin instance. Every hook below appends one entry to it as the system runs. Nothing is written to disk until the very end.

| Hook | Fires when | What we log |
|---|---|---|
| `before_agent_callback` | Any agent starts executing | agent name, timestamp |
| `after_agent_callback` | Any agent finishes executing | agent name, timestamp |
| `before_tool_callback` | Any tool is about to be called | tool name, arguments |
| `after_run_callback` | The entire run completes (Plugin-only hook) | flushes the log to disk |

`after_run_callback` does not exist on agent callbacks. A callback is scoped to one specific agent, so it has no notion of "the whole run is done." A Plugin sits above every agent, so it is the only place that concept can live. That is why the audit log gets flushed here, once, after everything else has finished.

In [10]:
import json
from datetime import datetime
from google.adk.plugins import BasePlugin

class AuditPlugin(BasePlugin):
    """
    A system-wide audit plugin that logs every agent invocation,
    tool call, and run completion across all agents in the App.
    Registered once on the App, it covers all 16 agents automatically.
    """

    def __init__(self):
        super().__init__(name="audit_plugin")
        self.log = []

    async def before_agent_callback(self, *, agent, callback_context):
        self.log.append({
            "timestamp": datetime.now().isoformat(),
            "event":     "agent_start",
            "agent":     agent.name,
        })
        return None

    async def after_agent_callback(self, *, agent, callback_context):
        self.log.append({
            "timestamp": datetime.now().isoformat(),
            "event":     "agent_end",
            "agent":     agent.name,
        })
        return None

    async def before_tool_callback(self, *, tool, tool_args, tool_context):
        self.log.append({
            "timestamp": datetime.now().isoformat(),
            "event":     "tool_call",
            "tool":      tool.name,
            "args":      str(tool_args),
        })
        return None

    async def after_run_callback(self, *, invocation_context):
        with open("audit_trail.json", "w") as f:
            json.dump(self.log, f, indent=2)
        print(f"✅ Audit trail saved — {len(self.log)} events logged")

## Cell 11 — Register the AuditPlugin on the App [LIVE]

`App` is the top-level container that binds `master_orchestrator_agent` to application-wide configuration, including plugins. Plugins go on the `App`, not the `Runner`. `Runner(plugins=[...])` still runs, but it raises a `DeprecationWarning` and is not the pattern this course teaches.

Passing `plugins=[AuditPlugin()]` here is the entire integration. There is no per-agent wiring. `App.plugins` applies to every agent reachable from `root_agent`, which for this system means all 16 agents: the ten specialists, the five workflow agents, and `master_orchestrator_agent` itself.

In [11]:
from google.adk.apps import App

app = App(
    name="autonomous_event_genie",
    root_agent=master_orchestrator_agent,
    plugins=[AuditPlugin()]
)

## Cell 12 — Run the Four-Turn Conversation [LIVE]

This is the same four-turn conversation from Lecture 9.2, now running with `AuditPlugin` active. The session is created with `app_name=app.name` — the session service keys everything by that name and it must match the `App` object exactly. Each turn passes just the query, the session, and the user id to `run_agent_query`. The `app` object is closed over by the helper, so it does not appear in the call site.

Watch for the `✅ Audit trail saved` confirmation message at the end. That line only prints once, from inside `after_run_callback`, after the entire run has completed.

In [12]:
session = await session_service.create_session(
    app_name=app.name, user_id="student_user"
)

response_1 = await run_agent_query(
    "I need to plan a 50-person AI tech meetup in Austin, Texas with a budget of $4000. Find some vendors.",
    session,
    "student_user"
)
print(response_1)

response_2 = await run_agent_query(
    "This looks great. Please add 'Grace Hopper' to the guest list. Her email is grace@example.com.",
    session,
    "student_user"
)
print(response_2)

response_3 = await run_agent_query(
    "Now, please draft an announcement email based on the final plan.",
    session,
    "student_user"
)
print(response_3)

response_4 = await run_agent_query(
    "Perfect. Please send the email to the guest list.",
    session,
    "student_user"
)
print(response_4)

/usr/local/lib/python3.13/dist-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


✅ Audit trail saved — 31 events logged
✅ Audit trail saved — 32 events logged
Here is a complete event plan and vendor proposal for your **50-person AI Tech Meetup in Austin, Texas**, staying well within your **$4,000 budget**:

---

# 🚀 Event Plan: Austin AI Tech Meetup

### 📊 Executive Summary
* **Guest Count:** 50 attendees
* **Total Budget:** $4,000.00
* **Estimated Cost (Recommended Plan):** $2,025.00
* **Remaining Surplus:** **+$1,975.00**

---

### 🏢 1. Recommended Vendors

#### **Venue: Createscape Coworking (Study Hall & Event Space)**
* **Location:** 701 Tillery St, Austin, TX 78702 (East Austin)
* **Rate:** $175/hour (4-hour rental) → **$700.00**
* **Why it fits:** 
  * High-speed fiber internet ideal for technical demos and live coding presentations.
  * Built-in HD projector, sound system, and whiteboards.
  * Free on-site parking for all 50 attendees.

#### **Catering: Smokey Mo's BBQ**
* **Format:** Standard BBQ Buffet (3 meats, 3 sides, bread, BBQ sauce, pickles, onions

## Cell 13 — Inspect the Audit Trail [LIVE]

The four-turn conversation is done, and `audit_trail.json` now exists on disk. This cell loads it back and looks for three things:

1. **Parallel execution made visible.** `venue_scout_agent`, `catering_scout_agent`, and `entertainment_scout_agent` should show `agent_start` entries at almost identical timestamps. That is proof `parallel_logistics_scout` ran them simultaneously, not proof by assumption.
2. **The budget loop made visible.** `accountant_agent` and `cost_cutter_agent` entries should alternate. Counting the alternations tells you exactly how many loop iterations ran.
3. **Scale.** The total event count is everything that happened inside this system for one four-turn conversation, captured by roughly twenty lines of Plugin code.

In [13]:
import json

with open("audit_trail.json", "r") as f:
    audit_log = json.load(f)

print(f"Total events logged: {len(audit_log)}")
print("\nFirst 10 entries:")
for entry in audit_log[:10]:
    print(entry)

print("\nAgents that ran (unique):")
agents_ran = list(dict.fromkeys(e["agent"] for e in audit_log if "agent" in e))
for agent in agents_ran:
    print(f"  - {agent}")

Total events logged: 46

First 10 entries:
{'timestamp': '2026-09-01T12:22:18.425016', 'event': 'agent_start', 'agent': 'master_orchestrator_agent'}
{'timestamp': '2026-09-01T12:22:44.805150', 'event': 'tool_call', 'tool': 'full_plan_workflow', 'args': "{'request': 'Plan a 50-person AI tech meetup in Austin, Texas with a budget of $4000, including vendor search.'}"}
{'timestamp': '2026-09-01T12:22:44.805805', 'event': 'agent_start', 'agent': 'full_plan_workflow'}
{'timestamp': '2026-09-01T12:22:44.805868', 'event': 'agent_start', 'agent': 'intake_agent'}
{'timestamp': '2026-09-01T12:22:51.744122', 'event': 'tool_call', 'tool': 'update_session_state', 'args': "{'event_type': 'AI tech meetup', 'city': 'Austin, Texas', 'budget': 4000}"}
{'timestamp': '2026-09-01T12:22:57.380844', 'event': 'agent_end', 'agent': 'intake_agent'}
{'timestamp': '2026-09-01T12:22:57.381005', 'event': 'agent_start', 'agent': 'initial_planning_workflow'}
{'timestamp': '2026-09-01T12:22:57.381067', 'event': 'agent